In [ ]:
# loading libraries
import qiskit.qasm2
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import SwapGate
from qiskit.dagcircuit import DAGCircuit
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.visualization import circuit_drawer
from qiskit.transpiler import PassManager, Layout, generate_preset_pass_manager
from qiskit.transpiler.passes import TrivialLayout
from qiskit.transpiler.basepasses import TransformationPass
from qiskit.visualization.dag_visualization import dag_drawer

from tqdm import tqdm

from rustworkx.visualization import mpl_draw
from rustworkx import spring_layout

from pathlib import Path

from itertools import permutations
from math import pi

import os

# Initialization

## Preparing backend
We can use fake_adonis wich gives the same star shape 5 qubit architecture as ODRA5. As a step, we should code our own ODRA5 replica as a Target object.

In [ ]:
# preparing backends
from iqm.qiskit_iqm.fake_backends import fake_adonis
fakeBackend = fake_adonis.IQMFakeAdonis()

## Loading circuit and Layout
Circut can be created hier in pure python code or loaded from qasm file. Currently qasm 3.1 is the latest standard, but qasm 2.0 is still available. In our project we can use the 2.0 version, because there are more sources on the internet, and it should work just fine. Link to qasm instructions https://arxiv.org/abs/1707.03429v2

In [ ]:
# loading circuit frm qasm file (simple as that)
# circ = qiskit.qasm2.load("ghz_central.qasm")
# This is the same quantum circuit as in qasm2 file
q = QuantumRegister(5, "q")
c = ClassicalRegister(5, "c")
circ = QuantumCircuit(q, c)
circ.h(3)
circ.cx(4, 3)
circ.cx(0, 1)
circ.measure_all()

In [ ]:
# loading circuit from qasm file
circ = qiskit.qasm2.load("qft.qasm")

In [ ]:
#Saving circuit to qasm2 file
qiskit.qasm2.dump(circ, "complex-circuit.qasm")

In [ ]:
circ.draw(output="mpl")

# Routing

In [ ]:
# class for custom routing pass (stage)
class CustomRouting(TransformationPass):
    # constructor
    def __init__(self, coupling_map, initial_layout=None):
        super().__init__()
        self.coupling_map = coupling_map
        self.initial_layout = initial_layout
 
    # main pass (transpile stage) function
    def run(self, dag):

        # check for initial laqout and assume trivial layout if none was given
        # I don't think this is used in this version, but not sure if this is correct
        if self.initial_layout is None:
            self.initial_layout = Layout.generate_trivial_layout(
                *dag.qregs.values()
            )

        # keeping the best DAG and best number of 2-qubit gates
        best_dag = None
        best_num_gates = 0
        best_layout = None

        # create list of qubits for permuting (with optional printing)
        qubitList = []
        for key, qreg in dag.qregs.items():
            for qbit in range(len(qreg)):
                qubitList.append(qreg[qbit])

        # testing all permutations
        for p in permutations(qubitList):

            # building the layout based on the permutation
            qubitDict = {}
            for i in range(len(qubitList)):
                qubitDict[i] = p[i]
            current_layout = Layout()
            current_layout.from_dict(qubitDict)
            for qreg in dag.qregs.values():
                current_layout.add_register(qreg)

            # creating new DAG that will be our result (for this permutation)...
            # ...and filling it with all qubits and classical bits
            new_dag = DAGCircuit()
            for qreg in dag.qregs.values():
                new_dag.add_qreg(qreg)
            for creg in dag.cregs.values():
                new_dag.add_creg(creg)
    
            # for each DAG layer
            for layer in dag.serial_layers():
                # get the layer
                subdag = layer["graph"]
                # for all gates in the layer that are 2-qubit gates (this is before native gates...
                # ...so it doesn't have to be just CZ gates)
                for gate in subdag.two_qubit_ops():

                    # get the qubit arguments for the gate
                    q0, q1 = gate.qargs

                    # get the positions given the current (both in terms of current layer in the circuit...
                    # ...and current permutation) layout 
                    p0 = current_layout[q0]
                    p1 = current_layout[q1]
    
                    # if the qubits aren't neighbors, we'll have to insert swaps
                    if self.coupling_map.distance(p0, p1) != 1:
                        # find path to adjoin the qubits
                        path = self.coupling_map.shortest_undirected_path(p0, p1)
                        # for all elements of the path
                        for i in range(len(path) - 2):
                            # get relevant wires
                            wire1, wire2 = path[i], path[i + 1]
                            # get the relevant qubits from the wires
                            qubit1 = qubitList[wire1]
                            qubit2 = qubitList[wire2]
                            qubit1 = current_layout[wire1]
                            qubit2 = current_layout[wire2]

                            # adds swap gate to the end of the relevant wires
                            new_dag.apply_operation_back(
                                SwapGate(), qargs=[qubit1, qubit2]
                            )
                            # update the layout given the swapped wires
                            current_layout.swap(wire1, wire2)
                    
                    # else, the gate was single-qubit, no change needed

                # apply the layer to the original (per permutation) DAG, using the wires/qubits...
                # ...uses re-ordering, but I don't understand this fully
                new_dag.compose(
                    subdag, qubits=current_layout.reorder_bits(new_dag.qubits)
                )

            # get number of 2-qubit gates (with optional printing, this is per permutation)
            num_gates = len(new_dag.two_qubit_ops())

            # update the best known dag and layout with the current if 2-qubit gate number is best known so far
            if best_dag is None or num_gates < best_num_gates:
                best_num_gates = num_gates
                best_dag = new_dag
                best_layout = current_layout.copy()
            
        # set the layout property to the chosen layout
        self.property_set["layout"] = best_layout

        # return the chosen DAG 
        return best_dag

# Transpilation

In [ ]:
# custom transpile function (for given backed and circuit)
def customTranspile(backend, circuit):
    # standard staged pass manager on optimization_level=2 and with given backend  
    masterManager = generate_preset_pass_manager(2, backend)

    # custom pass manager to replace standard layout stage pass
    customLayout = PassManager(
    [
        # use trivial layout (virtual 0 = physical 0, virtual 1 = physical 1 etc.) with the backend's coupling_map
        TrivialLayout(backend.coupling_map)
    ]
    )    
    # custom pass manager to replace standard routing stage pass with the backend's coupling_map
    customRouting = PassManager(
    [
        CustomRouting(backend.coupling_map, None)
    ]
    )

    # replace layout and routing stages in our custom manager
    # the first line is commented, so layout stage is done normally, this makes the code work for this example...
    # ...but it still breaks on different permutations and this shouldn't be commented anyway
    masterManager.layout = customLayout     
    masterManager.routing = customRouting

    # transpile the given circuit with our pass manager and return the transpiled circuit
    return masterManager.run(circuit)

# Output

In [ ]:
# draw input circuit (before transpilation)
circ.draw(output="mpl")

In [ ]:
# transpile the circuit with out custom manager and print the resulting circuit
transpiled = customTranspile(fakeBackend, circ)
transpiled.draw(output="mpl")

In [ ]:
# print the number of 2 qubit gates (i.e. CZ gates) after full transpilation
dag = circuit_to_dag(transpiled)
len(dag.two_qubit_ops())
# transpiled.layout.final_index_layout()

In [ ]:
print(transpiled.data)